<a href="https://colab.research.google.com/github/long7872/GNN/blob/main/Chapter%207/Section_7_5_Sparse_and_Dense_Graph_Representations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sparse vs Dense Graph Representations



This code snippet demonstrates the implementation of the GraphSAGE model with both dense and sparse data representations for node classification on the 'ogbn-products' dataset from the OGB (Open Graph Benchmark). (Spoiler: the dense representation will overwhelm the GPU, unless you have at least 50GB of usable GPU memory. The sparse representation takes up about half the GPU RAM.)

Two distinct GraphSAGE classes are defined for handling sparse and dense data. The code leverages pynvml to monitor GPU memory usage during model training and testing.

The user has the flexibility to choose between the sparse or dense representation by toggling the use_sparse variable. Correspondingly, different versions of the train and test functions are defined and selected to handle the selected data representation. The functions record and print out the loss, correctness, and GPU memory usage during execution.

The model, once trained, is tested, and the predictions, true labels, and memory usage are printed out. Finally, the pynvml library is shut down to free up resources.

## Overview of Graph Representations in Pytorch Geometric
###Sparse Graphs
In the context of graph neural networks, sparse graphs are typically represented using adjacency matrices. An adjacency matrix is a square matrix used to represent a finite graph. The elements of the matrix indicate whether pairs of vertices are adjacent or not in the graph. For large graphs, the adjacency matrix can be highly sparse, meaning that most of the elements are zero. In practice, we only store the non-zero elements to save memory and computational resources. This is particularly crucial for large-scale graphs like social networks, citation networks, etc., where the number of nodes can be in the millions or more.

###Dense Graphs
On the other hand, dense graphs or graph data represented in a dense format often use an edge list for representation. An edge list consists of pairs of nodes that have a direct edge between them. It’s a simple and explicit way to represent a graph but can be inefficient for large, sparse graphs, as it doesn't exploit the sparsity of the data.

###Differences in GraphSAGE Classes
#####Input Data Format:

GraphSAGE_sparse: Takes adjacency matrices as input. The forward method accepts adj_t (a sparse tensor representation of the adjacency matrix).

GraphSAGE_dense: Takes edge lists as input. The forward method accepts edge_index that provides the indices of the source and target nodes for each edge.
#####Memory Efficiency:

GraphSAGE_sparse: More memory-efficient for large, sparse graphs as it only stores non-zero elements of the adjacency matrix.

GraphSAGE_dense: Can be less efficient in terms of memory for large, sparse graphs but is suitable for smaller or denser graphs.
#####Computational Efficiency:

GraphSAGE_sparse: Can be computationally efficient for certain operations due to the sparse nature of the data.

GraphSAGE_dense: May involve more computations as it deals with the entire edge list or adjacency matrix.
Code Adaptability:

The two classes ensure that the GraphSAGE model can be easily adapted to different data formats and storage requirements, offering flexibility to the user.

In [1]:
#Uninstall the current CUDA version
!apt-get --purge remove cuda nvidia* libnvidia-*
!dpkg -l | grep cuda- | awk '{print $2}' | xargs -n1 dpkg --purge
!apt-get remove cuda-*
!apt autoremove
!apt-get update

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'nvidia-headless-no-dkms-615-server' for glob 'nvidia*'
Note, selecting 'nvidia-kernel-common-595-server' for glob 'nvidia*'
Note, selecting 'nvidia-dkms-580-server' for glob 'nvidia*'
Note, selecting 'nvidia-firmware-615-615.71.09' for glob 'nvidia*'
Note, selecting 'nvidia-driver-550-server' for glob 'nvidia*'
Note, selecting 'nvidia-firmware-550-server-550.144.03' for glob 'nvidia*'
Note, selecting 'nvidia-kernel-source-575-open' for glob 'nvidia*'
Note, selecting 'nvidia-firmware-595-595.91.07' for glob 'nvidia*'
Note, selecting 'nvidia-driver-570-server' for glob 'nvidia*'
Note, selecting 'nvidia-cuda-toolkit-doc' for glob 'nvidia*'
Note, selecting 'nvidia-kernel-common-615-server' for glob 'nvidia*'
Note, selecting 'nvidia-imex' for glob 'nvidia*'
Note, selecting 'nvidia-driver-590-server' for glob 'nvidia*'
Note, selecting 'nvidia-headless-390' for glob 'nvidia*'
Note

In [2]:

#Download CUDA 11.6
!wget  --no-clobber https://developer.download.nvidia.com/compute/cuda/11.6.0/local_installers/cuda-repo-ubuntu1804-11-6-local_11.6.0-510.39.01-1_amd64.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/cuda-repo-ubuntu1804_10.0.130-1_amd64.deb
#install CUDA kit dpkg
!dpkg -i cuda-repo-ubuntu1804-11-6-local_11.6.0-510.39.01-1_amd64.deb
!sudo apt-key adv --fetch-keys https://developer.download.nvidia.com/compute/cuda-repo-ubuntu1804-11-6-local/7fa2af80.pub
!apt-get update
!apt-get install cuda-11-6

File ‘cuda-repo-ubuntu1804-11-6-local_11.6.0-510.39.01-1_amd64.deb’ already there; not retrieving.

File ‘cuda-repo-ubuntu1804_10.0.130-1_amd64.deb’ already there; not retrieving.

Selecting previously unselected package cuda-repo-ubuntu1804-11-6-local.
(Reading database ... 122961 files and directories currently installed.)
Preparing to unpack cuda-repo-ubuntu1804-11-6-local_11.6.0-510.39.01-1_amd64.deb ...
Unpacking cuda-repo-ubuntu1804-11-6-local (11.6.0-510.39.01-1) ...
Setting up cuda-repo-ubuntu1804-11-6-local (11.6.0-510.39.01-1) ...

The public CUDA GPG key does not appear to be installed.
To install the key, run this command:
sudo apt-key add /var/cuda-repo-ubuntu1804-11-6-local/7fa2af80.pub

Executing: /tmp/apt-key-gpghome.bnuAmt4Y2o/gpg.1.sh --fetch-keys https://developer.download.nvidia.com/compute/cuda-repo-ubuntu1804-11-6-local/7fa2af80.pub
gpg: requesting key from 'https://developer.download.nvidia.com/compute/cuda-repo-ubuntu1804-11-6-local/7fa2af80.pub'
gpg: WARNING: u

In [3]:
!pip uninstall torch -y

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0


In [4]:
version = "2.11.0+cu130"
!pip install torch==2.11.0 -f https://data.pyg.org/whl/torch-{version}.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
  Using cached torch-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
Using cached torch-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl (530.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.29.1+cu130 requires torch>=2.14.0, but you have torch 2.11.0 which is incompatible.


In [5]:
!pip install psutil

In [6]:
# Find the CUDA version PyTorch was installed with
!python -c "import torch; print(torch.version.cuda)"

/usr/local/lib/python3.13/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
13.0


In [7]:
# PyTorch version
!python -c "import torch; print(torch.__version__)"

/usr/local/lib/python3.13/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2.11.0+cu130


In [17]:

!pip install pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{version}.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html


313pm

In [18]:
!pip install ogb
!pip install pynvml

In [19]:
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv
from torch_geometric.transforms import ToSparseTensor
from ogb.nodeproppred import PygNodePropPredDataset
import numpy as np
import pynvml

We use pynvml to get the GPU memory usage. This function can be placed anywhere in the code. In the example below, it is placed after the epoch. When I trouble shoot memory issues, one simple way to profile the code is to place this function in serveral spots in the code, as one would with debugging print statements.


In [20]:
# Initialize NVML
pynvml.nvmlInit()

# Function to get the current GPU memory usage using pynvml
def get_gpu_memory_usage():
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # 0 is GPU index
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return info.used / 1024 / 1024  # convert to MB

Definition of the GraphSAGE model for sparse data. These layers use adjacency matrices.

In [21]:
class GraphSAGE_sparse(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, out_dim)
    def forward(self, x, adj_t):
        x = self.conv1(x, adj_t)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout)
        x = self.conv2(x, adj_t)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout)
        x = self.conv3(x, adj_t)
        return torch.log_softmax(x, dim=-1)

Definition of the GraphSAGE model for dense data. These layers use edge indices.

In [22]:
class GraphSAGE_dense(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, out_dim)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout)
        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout)
        x = self.conv3(x, edge_index)
        return torch.log_softmax(x, dim=-1)

Train and test functions for the sparse representation. In this simple example, there is only one epoch.



In [23]:
def train_sparse(model, data, optimizer):
    model.train()
    data = data.to('cuda')

    optimizer.zero_grad()
    out = model(data.x, data.adj_t)  # Use adj_t for sparse tensor representation
    loss = F.cross_entropy(out, data.y.squeeze(1).long())
    loss.backward()
    optimizer.step()

    memory_usage = get_gpu_memory_usage()  # Getting memory usage with pynvml

    correct = (out.argmax(dim=1) == data.y.squeeze(1)).sum().item()
    return loss.item(), correct, memory_usage

@torch.no_grad()
def test_sparse(model, data):
    model.eval()
    data = data.to('cuda')

    out = model(data.x, data.adj_t)  # Use adj_t for sparse tensor representation
    pred = out.argmax(dim=1)

    memory_usage = get_gpu_memory_usage()  # Getting memory usage with pynvml

    return pred.cpu(), data.y.cpu(), memory_usage

Training/Test functions for dense. Again, we opt for one epoch in this example.

In [24]:
def train_dense(model, data, optimizer):
    model.train()
    data = data.to('cuda')

    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out, data.y.squeeze(1).long())
    loss.backward()
    optimizer.step()

    memory_usage = get_gpu_memory_usage()  # Getting memory usage with pynvml

    correct = (out.argmax(dim=1) == data.y.squeeze(1)).sum().item()
    return loss.item(), correct, memory_usage

@torch.no_grad()
def test_dense(model, data):
    model.eval()
    data = data.to('cuda')

    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    memory_usage = get_gpu_memory_usage()  # Getting memory usage with pynvml

    return pred.cpu(), data.y.cpu(), memory_usage

In [25]:
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

# Choose either sparse or dense
use_sparse = True  # Set to True for sparse, False for dense

torch.serialization.add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])

if use_sparse:
    # Load the dataset
    dataset = PygNodePropPredDataset(name='ogbn-products', transform=T.ToSparseTensor())
    data = dataset[0]
    model = GraphSAGE_sparse(dataset.num_features, 128, dataset.num_classes)
    train = train_sparse
    test = test_sparse
else:
    # Load the dataset
    dataset = PygNodePropPredDataset(name='ogbn-products')
    data = dataset[0]
    model = GraphSAGE_dense(dataset.num_features, 128, dataset.num_classes)
    train = train_dense
    test = test_dense

In our scenario, opting for a sparse representation should work fine. Opting for a dense representation of the data will lead to an `OutOfMemoryError`.

This error occurs when the GPU runs out of memory during the training process, which is attributed to the large memory requirement of handling dense data representations. The attempted allocation of 46.09 GiB of memory surpasses the total GPU capacity, causing the error.

One effective strategy to mitigate such memory issues is implementing mini-batching. Mini-batching breaks down the dataset into smaller, manageable batches, reducing the amount of memory needed at any given time. In the context of the Section 8.7 notebook, mini-batching is successfully applied to handle the same dense representation without overwhelming the memory, thus preventing the `OutOfMemoryError`.

In [26]:
model = model.to('cuda')
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training and testing the model
loss, correct, train_memory = train(model, data, optimizer)
print(f"Loss: {loss}, Correct: {correct}, Memory Used: {train_memory} MB")

pred, true_labels, test_memory = test(model, data)
print(f"Predictions: {pred}, True Labels: {true_labels}, Memory Used: {test_memory} MB")

# Shutdown NVML
pynvml.nvmlShutdown()

Loss: 3.9312806129455566, Correct: 27070, Memory Used: 22705.1875 MB
Predictions: tensor([17,  1,  2,  ...,  8,  2,  4]), True Labels: tensor([[0],
        [1],
        [2],
        ...,
        [8],
        [2],
        [4]]), Memory Used: 22705.1875 MB
